# Предобработка

Пайплайн обогащения таблицы `flats` в PostgreSQL
Этапы: нормализация -> вычисления -> кросс по домам -> геообогащение -> статистическая импутация.

In [ ]:
import psycopg2
import pandas as pd
import numpy as np

from config.settings import DB_CONFIG

def run_sql(query, fetch=False):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute(query)
    result = None
    if fetch:
        cols = [d[0] for d in cur.description]
        result = pd.DataFrame(cur.fetchall(), columns=cols)
    else:
        print(f"OK: {cur.statusmessage}")
    conn.commit()
    cur.close()
    conn.close()
    return result

def read_sql(query):
    conn = psycopg2.connect(**DB_CONFIG)
    df = pd.read_sql(query, conn)
    conn.close()
    return df

In [ ]:
# текущее состояние пропусков -- baseline для сравнения
read_sql("""
    SELECT source,
      COUNT(*) as total,
      SUM(CASE WHEN price_per_m2 IS NULL OR price_per_m2 = 0 THEN 1 ELSE 0 END) as no_ppm2,
      SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) as no_region,
      SUM(CASE WHEN year_built IS NULL THEN 1 ELSE 0 END) as no_year,
      SUM(CASE WHEN building_type IS NULL THEN 1 ELSE 0 END) as no_btype,
      SUM(CASE WHEN metro_stations IS NULL THEN 1 ELSE 0 END) as no_metro,
      SUM(CASE WHEN living_area IS NULL THEN 1 ELSE 0 END) as no_living,
      SUM(CASE WHEN rooms IS NULL THEN 1 ELSE 0 END) as no_rooms
    FROM flats GROUP BY source ORDER BY total DESC
""")

## 1. Нормализация building_type

Объединяем синонимы (Монолит -> Монолитный, и тд), "Другое" -> NULL (это код 0 = не указано).

In [ ]:
BTYPE_NORM = {
    "Монолит": "Монолитный",
    "Кирпич": "Кирпичный",
    "Панель": "Панельный",
    "Блоки": "Блочный",
    "Дерево": "Деревянный",
}

for old, new in BTYPE_NORM.items():
    run_sql(f"UPDATE flats SET building_type = '{new}' WHERE building_type = '{old}'")

run_sql("UPDATE flats SET building_type = NULL WHERE building_type = 'Другое'")

# перевод renovation в egorkainov
run_sql("""
    UPDATE flats SET renovation = 'Дизайнерский'
    WHERE source = 'kaggle_egorkainov' AND renovation = 'Designer'
""")

# проверка
read_sql("""
    SELECT building_type, COUNT(*) as n
    FROM flats WHERE building_type IS NOT NULL
    GROUP BY building_type ORDER BY n DESC
""")

## 2. Вычисление price_per_m2

In [ ]:
run_sql("""
    UPDATE flats
    SET price_per_m2 = (price / total_area)::bigint
    WHERE (price_per_m2 IS NULL OR price_per_m2 = 0)
      AND total_area IS NOT NULL AND total_area > 0
      AND price IS NOT NULL AND price > 0
""")

## 3. Кросс-заполнение year_built по координатам дома

Один дом = координаты, округлённые до 4 знаков (~11м). Если в доме хотя бы одна квартира с year_built -- заполняем остальные.

In [ ]:
# year_built: MODE() по дому
run_sql("""
    UPDATE flats f SET year_built = sub.yr
    FROM (
        SELECT ROUND(lat::numeric, 4) as lat_r, ROUND(lon::numeric, 4) as lon_r,
               MODE() WITHIN GROUP (ORDER BY year_built) as yr
        FROM flats
        WHERE year_built IS NOT NULL AND lat IS NOT NULL AND lat != 0
        GROUP BY lat_r, lon_r
    ) sub
    WHERE f.year_built IS NULL
      AND f.lat IS NOT NULL AND f.lat != 0
      AND ROUND(f.lat::numeric, 4) = sub.lat_r
      AND ROUND(f.lon::numeric, 4) = sub.lon_r
""")

# building_type: аналогично
run_sql("""
    UPDATE flats f SET building_type = sub.bt
    FROM (
        SELECT ROUND(lat::numeric, 4) as lat_r, ROUND(lon::numeric, 4) as lon_r,
               MODE() WITHIN GROUP (ORDER BY building_type) as bt
        FROM flats
        WHERE building_type IS NOT NULL AND lat IS NOT NULL AND lat != 0
        GROUP BY lat_r, lon_r
    ) sub
    WHERE f.building_type IS NULL
      AND f.lat IS NOT NULL AND f.lat != 0
      AND ROUND(f.lat::numeric, 4) = sub.lat_r
      AND ROUND(f.lon::numeric, 4) = sub.lon_r
""")

## 4. Геообогащение

Определяем округ, район, ближайшее метро и расстояние до центра по координатам.
Нужны: `geopandas`, `shapely`, `scipy`, GeoJSON-полигоны округов/районов, справочник метро.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
from math import radians, sin, cos, sqrt, atan2
from psycopg2.extras import Json


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

### 4a. Загрузка справочных данных

Полигоны округов и районов Москвы, справочник метро.
GeoJSON скачиваем из открытых источников (OSM / data.mos.ru).

In [ ]:
okrugs = gpd.read_file("../data/moscow_admin_okrugs.geojson")
districts = gpd.read_file("../data/moscow_districts.geojson")

print(f"Округов: {len(okrugs)}, районов: {len(districts)}")
okrugs.head()

### 4b. Округ и район по координатам (point-in-polygon)

In [ ]:
# выгружаем записи с координатами и без региона
pts = read_sql("""
    SELECT id, lat, lon FROM flats
    WHERE lat IS NOT NULL AND lat != 0 AND region IS NULL
""")
print(f"Записей для геокодинга: {len(pts)}")

geometry = [Point(lon, lat) for lat, lon in zip(pts["lat"], pts["lon"])]
gdf = gpd.GeoDataFrame(pts, geometry=geometry, crs="EPSG:4326")

# spatial join с округами
# колонка с названием округа зависит от структуры GeoJSON -- подставить нужное имя
# okrugs должен иметь колонку с названием (напр. "NAME" или "name" или "ABBREV")
joined = gpd.sjoin(gdf, okrugs[["geometry", "NAME"]], predicate="within", how="left")
joined = joined.rename(columns={"NAME": "okrug"})

# spatial join с районами
joined2 = gpd.sjoin(gdf, districts[["geometry", "NAME"]], predicate="within", how="left")
joined2 = joined2.rename(columns={"NAME": "rayon"})

pts["region"] = joined["okrug"].values
pts["district"] = joined2["rayon"].values

filled = pts.dropna(subset=["region"])
print(f"Определён округ: {len(filled)} из {len(pts)}")
filled[["region", "district"]].head(10)

In [ ]:
# записываем region и district обратно в БД батчами
BATCH = 10_000
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

updated = 0
for i in range(0, len(filled), BATCH):
    batch = filled.iloc[i:i + BATCH]
    for _, row in batch.iterrows():
        cur.execute(
            "UPDATE flats SET region = %s, district = %s WHERE id = %s",
            (row["region"], row.get("district"), int(row["id"])),
        )
    conn.commit()
    updated += len(batch)
    print(f"  {updated}/{len(filled)}")

cur.close()
conn.close()
print(f"Обновлено: {updated}")

### 4c. Ближайшее метро по координатам

In [ ]:
import json

with open("../data/moscow_metro_stations.json", "r", encoding="utf-8") as f:
    metro_raw = json.load(f)

# ожидаемый формат: [{"name": "Павелецкая", "lat": 55.7298, "lon": 37.6364}, ...]
metro_df = pd.DataFrame(metro_raw)
print(f"Станций метро: {len(metro_df)}")

metro_coords = np.radians(metro_df[["lat", "lon"]].values)
tree = cKDTree(metro_coords)

In [ ]:
# записи без метро и с координатами
no_metro = read_sql("""
    SELECT id, lat, lon FROM flats
    WHERE metro_stations IS NULL AND lat IS NOT NULL AND lat != 0
""")
print(f"Записей без метро: {len(no_metro)}")

flat_coords = np.radians(no_metro[["lat", "lon"]].values)
R_EARTH = 6371_000  # метры

# k=3 ближайших станции
dists, idxs = tree.query(flat_coords, k=3)

# конвертация угловых расстояний в метры (~точно для малых расстояний)
dists_m = dists * R_EARTH

results = []
for i in range(len(no_metro)):
    stations = []
    for j in range(3):
        name = metro_df.iloc[idxs[i][j]]["name"]
        walk_min = int(round(dists_m[i][j] / 80))  # ~80 м/мин пешком
        stations.append({"name": name, "minutes": walk_min})
    results.append(stations)

no_metro["metro_stations"] = results
no_metro.head()

In [ ]:
# запись metro_stations в БД
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

for i in range(0, len(no_metro), BATCH):
    batch = no_metro.iloc[i:i + BATCH]
    for _, row in batch.iterrows():
        cur.execute(
            "UPDATE flats SET metro_stations = %s WHERE id = %s",
            (Json(row["metro_stations"]), int(row["id"])),
        )
    conn.commit()
    print(f"  metro: {min(i + BATCH, len(no_metro))}/{len(no_metro)}")

cur.close()
conn.close()

## 5. Статистическая импутация

Заполняем living_area и kitchen_area медианными пропорциями по (building_type, rooms).
rooms -- грубая оценка по total_area.

In [ ]:
# rooms: грубая оценка по площади (где rooms IS NULL)
run_sql("""
    UPDATE flats SET rooms = CASE
        WHEN total_area < 35 THEN 0
        WHEN total_area < 50 THEN 1
        WHEN total_area < 75 THEN 2
        WHEN total_area < 100 THEN 3
        ELSE 4
    END
    WHERE rooms IS NULL AND total_area IS NOT NULL
""")

In [ ]:
# living_area и kitchen_area: медианные пропорции по (building_type, rooms)
ratios = read_sql("""
    SELECT building_type, rooms,
      PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY living_area / NULLIF(total_area, 0)) as live_r,
      PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY kitchen_area / NULLIF(total_area, 0)) as kitch_r,
      COUNT(*) as n
    FROM flats
    WHERE living_area IS NOT NULL AND total_area > 0 AND rooms >= 0
    GROUP BY building_type, rooms
    HAVING COUNT(*) > 50
""")
print(f"Групп с пропорциями: {len(ratios)}")
ratios.head(10)

In [ ]:
# применяем пропорции
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

filled_live, filled_kitch = 0, 0
for _, r in ratios.iterrows():
    bt, rooms = r["building_type"], int(r["rooms"])

    # building_type может быть NULL
    bt_cond = "building_type IS NULL" if pd.isna(bt) else f"building_type = '{bt}'"

    if pd.notna(r["live_r"]) and r["live_r"] > 0:
        cur.execute(f"""
            UPDATE flats SET living_area = ROUND((total_area * {r['live_r']})::numeric, 1)
            WHERE living_area IS NULL AND total_area > 0
              AND {bt_cond} AND rooms = {rooms}
        """)
        filled_live += cur.rowcount

    if pd.notna(r["kitch_r"]) and r["kitch_r"] > 0:
        cur.execute(f"""
            UPDATE flats SET kitchen_area = ROUND((total_area * {r['kitch_r']})::numeric, 1)
            WHERE kitchen_area IS NULL AND total_area > 0
              AND {bt_cond} AND rooms = {rooms}
        """)
        filled_kitch += cur.rowcount

conn.commit()
cur.close()
conn.close()
print(f"living_area: {filled_live}, kitchen_area: {filled_kitch}")

## 6. Верификация

Сравниваем пропуски до и после обогащения.

In [ ]:
# итоговые пропуски
read_sql("""
    SELECT source,
      COUNT(*) as total,
      SUM(CASE WHEN price_per_m2 IS NULL OR price_per_m2 = 0 THEN 1 ELSE 0 END) as no_ppm2,
      SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) as no_region,
      SUM(CASE WHEN district IS NULL THEN 1 ELSE 0 END) as no_district,
      SUM(CASE WHEN year_built IS NULL THEN 1 ELSE 0 END) as no_year,
      SUM(CASE WHEN building_type IS NULL THEN 1 ELSE 0 END) as no_btype,
      SUM(CASE WHEN metro_stations IS NULL THEN 1 ELSE 0 END) as no_metro,
      SUM(CASE WHEN living_area IS NULL THEN 1 ELSE 0 END) as no_living,
      SUM(CASE WHEN rooms IS NULL THEN 1 ELSE 0 END) as no_rooms
    FROM flats GROUP BY source ORDER BY total DESC
""")

In [ ]:
# spot-check: случайные записи с заполненным регионом
read_sql("""
    SELECT id, source, city, region, district, lat, lon, price, total_area, rooms,
           year_built, building_type, metro_stations::text
    FROM flats
    WHERE source = 'kaggle_mrdaniilak' AND region IS NOT NULL
    ORDER BY random() LIMIT 10
""")